# 05 - Report


In [ ]:
# test cell because the ydata_profiling refuses to work
import sys, ydata_profiling
print(sys.executable, ydata_profiling.__version__)

In [ ]:
## Testing if the data file exists in the correct location
import os

print(os.getcwd())
print(os.path.exists("../../data/melbourne/melb_data.csv"))
print(os.listdir("../.."))

In [ ]:
# This cell has been added after the cell below threw an error that my .venv file was still on the old one, and that matplotlib in the old venv was half-installed. 
# What I did was, i switched to the short-path venv and left the old one alone. Additionally left out dython on purpose since I assume it was causing conflict with matplotlib.
import sys; print(sys.executable)

# Im still trying to get to the bottom of this problem, hold on

In [ ]:
# ---- Setup: imports, data loading, datatype fixes (the only "fix" the task allows) ----
import sys, warnings

warnings.filterwarnings("ignore")
sys.path.append("../../")  # helper.py and report_tools.py live in project_1/
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from IPython.display import display
from helper import *  # course helper functions (column_report, vif_table, ...)
from report_tools import *  # small extra helpers, see AI disclosure

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

TARGET = "Price"
raw = pd.read_csv(
    "../../data/melbourne/melb_data.csv"
)  # untouched copy, used for "before" views
df = raw.copy()
df["Date"] = pd.to_datetime(
    df["Date"], dayfirst=True
)  # "3/12/2016" = 3 Dec 2016 (Australian format)
df["Postcode"] = df["Postcode"].astype("Int64").astype(str)  # a code, not an amount
num_cols, cat_cols = split_columns(df, TARGET)  # feature lists without the target
print(
    df.shape,
    "| numeric features:",
    len(num_cols),
    "| categorical features:",
    len(cat_cols),
)

## 1a) Datatypes

In [ ]:
# Step 1a: datatypes before and after the fixes
dtypes = pd.DataFrame(
    {
        "raw dtype": raw.dtypes.astype(str),
        "fixed dtype": df.dtypes.astype(str),
        "example value": raw.iloc[0].astype(str),
    }
)
display(dtypes)

## 1b) Shape

In [ ]:
# Step 1b: shape, and what happens to the shape under the obvious preparation steps
n_rows, n_cols = df.shape
levels = {c: df[c].nunique() for c in cat_cols}
print(f"Rows: {n_rows} | columns: {n_cols} (1 target, {n_cols - 1} features)")
print(f"Rows per raw feature: {n_rows / (n_cols - 1):.0f}")
print(
    "Columns after one-hot encoding every text column:",
    len(num_cols) + sum(levels.values()),
)
print(
    "  ...without Address:",
    len(num_cols) + sum(v for k, v in levels.items() if k != "Address"),
)
scenarios = pd.Series(
    {
        "all rows": len(df),
        "drop rows with any NaN": len(df.dropna()),
        "drop BuildingArea + YearBuilt, then rows with NaN": len(
            df.drop(columns=["BuildingArea", "YearBuilt"]).dropna()
        ),
    }
)
ax = scenarios.plot.barh(figsize=(8, 2.6), color=["steelblue", "tomato", "orange"])
for i, v in enumerate(scenarios):
    ax.text(v, i, f" {v:,}", va="center")
ax.set_title("Rows left after simple missing-value strategies")
ax.set_xlim(0, 16000)
plt.show()
print(scenarios.to_dict())

## 1c) Missing values, duplicates, zeros, impossible values

In [ ]:
# Step 1c-1: missing values (counts, pattern, and WHEN they are missing)
miss = (
    column_report(df)[["missing", "missing_%"]]
    .query("missing > 0")
    .sort_values("missing", ascending=False)
)
display(miss)
m = df[["BuildingArea", "YearBuilt"]].isna()
print("Total missing cells:", int(df.isna().sum().sum()))
print(
    "BuildingArea missing % by Type:",
    df.groupby("Type")["BuildingArea"]
    .apply(lambda s: round(s.isna().mean() * 100, 1))
    .to_dict(),
)
known = df.dropna(subset=["CouncilArea"])
print(
    f"Rows with missing CouncilArea whose Suburb appears elsewhere with a council: {df.loc[df.CouncilArea.isna(), 'Suburb'].isin(known.Suburb).mean():.1%}"
    f" | suburbs that map to more than one council: {(known.groupby('Suburb')['CouncilArea'].nunique() > 1).sum()}"
)
print(
    "BuildingArea AND YearBuilt both missing:",
    int(m.all(axis=1).sum()),
    "| only BuildingArea:",
    int((m.BuildingArea & ~m.YearBuilt).sum()),
    "| only YearBuilt:",
    int((~m.BuildingArea & m.YearBuilt).sum()),
)
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
miss["missing_%"].plot.bar(ax=ax[0], color="steelblue", rot=0)
ax[0].set_title("Missing values per column (%)")
month = df["Date"].dt.to_period("M")
df.groupby(month)["CouncilArea"].apply(lambda s: s.isna().mean() * 100).plot(
    ax=ax[1], marker="o", color="tomato"
)
ax[1].set_title("CouncilArea: % missing by month of sale")
ax[1].set_ylabel("% missing")
plt.tight_layout()
plt.show()

In [ ]:
# Step 1c-2: duplicates. Address alone is NOT an ID - the same street address exists in different suburbs.
print("Exact duplicate rows:", int(df.duplicated().sum()))
same = df[df.duplicated(["Address", "Date"], keep=False)].sort_values(
    ["Address", "Date"]
)
display(
    same[["Suburb", "Address", "Date", "Rooms", "Type", "Price", "SellerG", "Method"]]
)
resold = df.duplicated(["Suburb", "Address"], keep=False)
print(
    "Rows whose Suburb+Address appears more than once (re-sales / re-listings):",
    int(resold.sum()),
)

In [ ]:
# Step 1c-3: zeros, and whether they are real values or "missing in disguise"
zero_cols = ["Landsize", "Car", "Bathroom", "BuildingArea", "Bedroom2", "Distance"]
display(
    pd.DataFrame(
        {
            "zeros": (df[zero_cols] == 0).sum(),
            "zeros_%": ((df[zero_cols] == 0).mean() * 100).round(2),
        }
    )
)
display(
    pd.crosstab(df["Type"], df["Landsize"] == 0, margins=True).rename(
        columns={False: "Landsize > 0", True: "Landsize = 0"}
    )
)

In [ ]:
# Step 1c-4: values that make no sense (rule-based checks)
sale_year = df["Date"].dt.year
checks = pd.Series(
    {
        "YearBuilt < 1800": (df["YearBuilt"] < 1800).sum(),
        "YearBuilt after the sale year": (df["YearBuilt"] > sale_year).sum(),
        "BuildingArea < 20 m2 (incl. 0)": (df["BuildingArea"] < 20).sum(),
        "BuildingArea > Landsize (houses, land > 0)": (
            (df["BuildingArea"] > df["Landsize"])
            & (df["Landsize"] > 0)
            & (df["Type"] == "h")
        ).sum(),
        "Unit with Landsize > 1000 m2": (
            (df["Type"] == "u") & (df["Landsize"] > 1000)
        ).sum(),
        "Bedroom2 != Rooms": (df["Bedroom2"] != df["Rooms"]).sum(),
        "Bedroom2 > Rooms + 3": (df["Bedroom2"] > df["Rooms"] + 3).sum(),
        "Bathroom > Rooms": (df["Bathroom"] > df["Rooms"]).sum(),
        "Price is not a confirmed sale (Method PI or VB)": df["Method"]
        .isin(["PI", "VB"])
        .sum(),
    }
).to_frame("rows")
checks["% of rows"] = (checks["rows"] / len(df) * 100).round(2)
display(checks)
display(
    df.loc[
        (df["YearBuilt"] < 1800)
        | (df["Bedroom2"] > df["Rooms"] + 3)
        | (df["BuildingArea"] > 5000),
        [
            "Suburb",
            "Type",
            "Rooms",
            "Bedroom2",
            "Landsize",
            "BuildingArea",
            "YearBuilt",
            "Price",
        ],
    ]
)

## 1d) Alerts from big-picture tools

In [ ]:
# Step 1d: big-picture tools. Reports are written to project_1/reports/ (git-ignored html files).
alerts = try_ydata_profile(
    df,
    "Melbourne Housing - preliminary EDA",
    "../../reports/melbourne_ydata_profile.html",
)
if alerts is not None:
    display(alerts)
try_sweetviz(
    df.drop(columns=["Address"]), TARGET, "../../reports/melbourne_sweetviz.html"
)

In [ ]:
# The same alert rules as ydata-profiling, re-calculated with pandas (see report_tools.py).
# This runs everywhere, and is reused in Step 4d to check the tool.
yd = ydata_style_alerts(df)
display(yd)

In [ ]:
DF_AV, AV_DIR = df.drop(columns=["Address"]), "../../reports/autoviz_melbourne"
try:
    from autoviz.AutoViz_Class import AutoViz_Class

    AutoViz_Class().AutoViz(
        "",
        dfte=DF_AV,
        depVar=TARGET,
        chart_format="png",
        save_plot_dir=AV_DIR,
        verbose=0,
    )
except Exception as e:
    print(f"AutoViz skipped: {type(e).__name__}: {e}")

## 2a) Distributions and imbalances

In [ ]:
# Step 2a-1: the target
p = df[TARGET]
print(
    f"Price: skew {p.skew():.2f}, excess kurtosis {p.kurt():.2f} | log(Price): skew {np.log(p).skew():.2f}, kurtosis {np.log(p).kurt():.2f}"
)
print(
    f"min {p.min():,.0f}  median {p.median():,.0f}  mean {p.mean():,.0f}  max {p.max():,.0f}"
)
fig, ax = plt.subplots(1, 4, figsize=(20, 4))
sns.histplot(p, bins=60, kde=True, ax=ax[0])
ax[0].set_title("Price")
sns.histplot(np.log(p), bins=60, kde=True, ax=ax[1])
ax[1].set_title("log(Price) - diagnostic only")
stats.probplot(p, dist="norm", plot=ax[2])
ax[2].set_title("Q-Q: Price")
stats.probplot(np.log(p), dist="norm", plot=ax[3])
ax[3].set_title("Q-Q: log(Price)")
plt.tight_layout()
plt.show()

In [ ]:
# Step 2a-2: numeric features
display(skew_kurt_table(df, num_cols))
hist_grid(df, num_cols, ncols=4)

In [ ]:
# Q-Q plots for the four most skewed / heaviest-tailed features (+ Rooms as a discrete example)
qq_grid(
    df, ["Landsize", "BuildingArea", "YearBuilt", "Distance", "Car", "Rooms"], ncols=3
)

In [ ]:
# Step 2a-3: balance of the categorical features (low-cardinality ones)
fig, axes = plt.subplots(1, 3, figsize=(20, 4.5))
for ax, col in zip(axes, ["Type", "Method", "Regionname"]):
    vc = df[col].value_counts()
    vc.plot.bar(ax=ax, color="steelblue", rot=35)
    ax.set_title(
        f"{col}: largest level {vc.iloc[0] / len(df):.0%}, smallest {vc.iloc[-1]} rows"
    )
plt.tight_layout()
plt.show()
print("Regionname rows:", df["Regionname"].value_counts().to_dict())

## 2b) Multicollinearity and redundancy

In [ ]:
# Step 2b-1: correlation matrix + VIF for numeric features
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    df[num_cols + [TARGET]].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    ax=ax,
)
ax.set_title("Pearson correlation (numeric features + Price)")
plt.show()
display(high_corr_pairs(df, num_cols, 0.7))
display(vif_table(df, num_cols))

In [ ]:
# Step 2b-2: is Bedroom2 just a copy of Rooms? And how redundant are the location columns?
print(f"Bedroom2 == Rooms in {(df['Bedroom2'] == df['Rooms']).mean():.1%} of rows")
display(pd.crosstab(df["Rooms"], df["Bedroom2"]))
loc = (
    pd.Series(
        {
            "Suburbs with exactly 1 Propertycount value": (
                df.groupby("Suburb")["Propertycount"].nunique() == 1
            ).mean(),
            "Suburbs with exactly 1 Regionname": (
                df.groupby("Suburb")["Regionname"].nunique() == 1
            ).mean(),
            "Suburbs with exactly 1 Postcode": (
                df.groupby("Suburb")["Postcode"].nunique() == 1
            ).mean(),
            "Suburbs with exactly 1 Distance value": (
                df.groupby("Suburb")["Distance"].nunique() == 1
            ).mean(),
            "Suburbs with at most 2 Distance values": (
                df.groupby("Suburb")["Distance"].nunique() <= 2
            ).mean(),
        }
    )
    .mul(100)
    .round(1)
    .to_frame("% of suburbs")
)
display(loc)

In [ ]:
try:
    from skfeature.function.similarity_based import fisher_score

    smp = df[num_cols + [TARGET]].dropna().sample(5000, random_state=42)
    scores = fisher_score.fisher_score(smp[num_cols].values, smp[TARGET].values)
    pd.Series(scores, num_cols).sort_values().plot.barh(
        figsize=(7, 4), title="Fisher score (skfeature)"
    )
    plt.show()
except Exception as e:
    print(f"Fisher score (skfeature) skipped: {type(e).__name__}: {e}")

## 2c) High cardinality

In [ ]:
# Step 2c: cardinality of the categorical columns
card = cardinality_table(df, cat_cols)
display(card)
fig, ax = plt.subplots(1, 3, figsize=(20, 4.5))
card["unique"].sort_values().plot.barh(ax=ax[0], color="steelblue", logx=True)
ax[0].set_title("Distinct values per text column (log scale)")
for a, col in zip(ax[1:], ["Suburb", "SellerG"]):
    vc = df[col].value_counts()
    a.plot(range(1, len(vc) + 1), vc.values, color="tomato")
    a.set_yscale("log")
    a.set_title(
        f"{col}: rows per level, sorted ({(vc < 5).sum()} levels have < 5 rows)"
    )
    a.set_xlabel("rank")
plt.tight_layout()
plt.show()
agents = df["SellerG"].value_counts()
display(
    agents[agents.index.str.contains("C21|Century|hockingstuart", case=False)].to_frame(
        "rows"
    )
)

## 3a) Major trends

In [ ]:
# Step 3a-1: numeric trends against Price (4000-row sample so the plots stay readable)
s = df.sample(4000, random_state=1)
fig, axes = plt.subplots(2, 3, figsize=(19, 9))
for ax, col in zip(
    axes.ravel(),
    ["Rooms", "Bathroom", "Distance", "YearBuilt", "BuildingArea", "Landsize"],
):
    d = s[s[col] < s[col].quantile(0.99)] if col in ("BuildingArea", "Landsize") else s
    if col == "YearBuilt":
        d = s[s[col] > 1800]  # hide the single year-1196 typo
    sns.scatterplot(data=d, x=col, y=TARGET, alpha=0.15, s=8, ax=ax)
    binned_trend(ax, d[col], d[TARGET])
    ax.set_title(
        f"{col} (red = median price per bin{', top 1% cut' if d is not s else ''})"
    )
plt.tight_layout()
plt.show()

In [ ]:
# Step 3a-2: categorical trends + time
fig, axes = plt.subplots(1, 3, figsize=(21, 5))
for ax, col in zip(axes[:2], ["Type", "Regionname"]):
    order = df.groupby(col)[TARGET].median().sort_values().index
    sns.boxplot(data=df, x=col, y=TARGET, order=order, ax=ax, fliersize=1)
    ax.set_yscale("log")
    ax.tick_params(axis="x", rotation=35)
    ax.set_title(f"Price by {col} (log scale)")
q = df.groupby(df["Date"].dt.to_period("Q"))[TARGET].agg(["median", "count"])
axes[2].bar(q.index.astype(str), q["median"], color="steelblue")
axes[2].set_ylim(800_000, 1_000_000)
for i, (m_, n_) in enumerate(zip(q["median"], q["count"])):
    axes[2].text(i, m_ + 3000, f"n={n_}", ha="center", fontsize=8)
axes[2].set_title("Median price by quarter of sale (y-axis starts at 800k)")
plt.tight_layout()
plt.show()
display(q.T)

In [ ]:
# Step 3a-3: the Distance trend depends on the property type (composition effect)
by_type = pd.DataFrame(
    {
        t: {
            "rows": int((df.Type == t).sum()),
            "Spearman(Distance, Price)": round(
                df.loc[df.Type == t, "Distance"].corr(
                    df.loc[df.Type == t, TARGET], method="spearman"
                ),
                3,
            ),
        }
        for t in ["h", "t", "u"]
    }
).T
print(
    "All types together:", round(df["Distance"].corr(df[TARGET], method="spearman"), 3)
)
display(by_type)
dist_band = pd.cut(
    df["Distance"],
    [-1, 3, 6, 10, 15, 50],
    labels=["0-3 km", "3-6", "6-10", "10-15", "15+ km"],
)
display(
    pd.crosstab(dist_band, df["Type"], normalize="index")
    .round(2)
    .rename_axis("share of sales")
)
display(
    df.groupby([dist_band, "Type"], observed=True)[TARGET]
    .median()
    .unstack()
    .round(-3)
    .rename_axis("median price")
)

In [ ]:
# Decision tree (depth 3) - which variables does a tree split on first?
from sklearn.tree import DecisionTreeRegressor, plot_tree

Xt = pd.get_dummies(df[num_cols + ["Type"]], dtype=float)
Xt = Xt.fillna(Xt.median())  # tree needs complete data; analysis only
tree = DecisionTreeRegressor(max_depth=3, random_state=0).fit(Xt, df[TARGET])
plt.figure(figsize=(24, 8))
plot_tree(
    tree,
    feature_names=list(Xt.columns),
    filled=True,
    impurity=False,
    precision=2,
    fontsize=9,
)
plt.title("Decision tree for Price (depth 3)")
plt.show()

## 3b) Overlapping data

In [ ]:
# Step 3b-1: overlap - how much do price ranges of different groups overlap?
fig, axes = plt.subplots(1, 2, figsize=(17, 4.5))
sns.kdeplot(
    data=df[df["Rooms"].between(2, 5)],
    x=TARGET,
    hue="Rooms",
    log_scale=True,
    common_norm=False,
    ax=axes[0],
    palette="viridis",
)
axes[0].set_title("Price distribution per number of Rooms (log x-axis)")
sns.scatterplot(
    data=s, x="Distance", y=TARGET, hue="Type", alpha=0.35, s=10, ax=axes[1]
)
axes[1].set_yscale("log")
axes[1].set_title("Price vs Distance coloured by Type")
plt.tight_layout()
plt.show()
band = df[df["Rooms"] == 3][TARGET].quantile([0.05, 0.5, 0.95])
print(
    f"3-room properties: 5% at {band.iloc[0]:,.0f}, median {band.iloc[1]:,.0f}, 95% at {band.iloc[2]:,.0f}"
)
print(
    f"Share of 2-room properties more expensive than the median 4-room property: {(df.loc[df.Rooms == 2, TARGET] > df.loc[df.Rooms == 4, TARGET].median()).mean():.1%}"
)

In [ ]:
# Step 3b-2: how much of Price can ONE feature explain? (R^2 of a straight line, and of a shallow tree)
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score

r2 = {}
for col in [
    "Rooms",
    "Bathroom",
    "Distance",
    "BuildingArea",
    "YearBuilt",
    "Landsize",
    "Lattitude",
    "Longtitude",
]:
    d = df[[col, TARGET]].dropna()
    r2[col] = {
        "linear R2": d.corr().iloc[0, 1] ** 2,
        "depth-4 tree R2 (5-fold CV)": cross_val_score(
            DecisionTreeRegressor(max_depth=4, random_state=0),
            d[[col]],
            d[TARGET],
            cv=5,
            scoring="r2",
        ).mean(),
    }
display(
    pd.DataFrame(r2)
    .T.round(3)
    .sort_values("depth-4 tree R2 (5-fold CV)", ascending=False)
)

## 3c) Outliers and noise

In [ ]:
# Step 3c-1: univariate outlier counts with three rules
cols = num_cols + [TARGET]
res = (
    iqr_outliers(df, cols).join(zscore_outliers(df, cols)).join(mad_outliers(df, cols))
)
display(
    res[
        [
            "n_outliers",
            "pct",
            "n_|z|>3",
            "n_MAD_outliers",
            "lower_fence",
            "upper_fence",
            "min",
            "max",
        ]
    ].rename(columns={"n_outliers": "IQR_outliers", "pct": "IQR_%"})
)

In [ ]:
# Step 3c-2: how extreme are they? boxplots on the original and log scale
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, col in zip(axes, [TARGET, "Landsize", "BuildingArea", "YearBuilt"]):
    if col == "YearBuilt":
        sns.boxplot(x=df[col], ax=ax, fliersize=2)
        ax.set_title(col)
    else:  # log axis: zeros cannot be shown, so they are left out
        sns.boxplot(x=df.loc[df[col] > 0, col], ax=ax, fliersize=2, log_scale=True)
        ax.set_title(f"{col} (log scale, {(df[col] == 0).sum()} zeros not shown)")
plt.tight_layout()
plt.show()
display(
    df.nlargest(5, "Landsize")[
        ["Suburb", "Type", "Rooms", "Landsize", "BuildingArea", "Price"]
    ]
)
display(df.nsmallest(5, "YearBuilt")[["Suburb", "Type", "YearBuilt", "Price"]])
# the most expensive sales, next to the median price of their suburb
top = df.nlargest(6, TARGET)[
    [
        "Suburb",
        "Type",
        "Rooms",
        "Bathroom",
        "BuildingArea",
        "YearBuilt",
        "Method",
        TARGET,
    ]
]
top["suburb median price"] = top["Suburb"].map(df.groupby("Suburb")[TARGET].median())
display(top)

In [ ]:
# Step 3c-3: multivariate outliers (rows that are unusual as a combination)
out = pd.DataFrame(
    {
        "isolation_forest": isolation_forest_flags(df, cols),
        "elliptic_envelope": elliptic_flags(df, cols),
    }
)
print(out.sum().to_dict(), "| flagged by both:", int(out.all(axis=1).sum()))
both = df[out.all(axis=1)]
prof = pd.DataFrame(
    {
        "flagged by both": both[
            ["Price", "Landsize", "Distance", "YearBuilt", "Rooms"]
        ].median(),
        "all rows": df[
            ["Price", "Landsize", "Distance", "YearBuilt", "Rooms"]
        ].median(),
    }
)
display(prof.round(1).rename_axis("median of"))
print(
    "Flagged rows with Landsize > 5000 m2:",
    int((both.Landsize > 5000).sum()),
    "| Distance > 30 km:",
    int((both.Distance > 30).sum()),
    "| types:",
    both.Type.value_counts().to_dict(),
)
display(
    both[
        [
            "Suburb",
            "Type",
            "Rooms",
            "Price",
            "Distance",
            "Landsize",
            "BuildingArea",
            "YearBuilt",
        ]
    ]
    .sort_values(TARGET, ascending=False)
    .head(8)
)

## 3d) Correlations and associations

In [ ]:
# Step 3d-1: linear (Pearson) vs rank (Spearman) correlation with Price.
# A big gap means the relationship exists but is hidden by outliers or is non-linear.
cmp = pd.DataFrame(
    {
        "Pearson": df[num_cols].corrwith(df[TARGET]),
        "Spearman": df[num_cols].corrwith(df[TARGET], method="spearman"),
    }
)
cmp["gap"] = cmp["Spearman"].abs() - cmp["Pearson"].abs()
display(cmp.sort_values("Spearman", key=abs, ascending=False).round(3))

In [ ]:
# Step 3d-2: Phi-K association matrix (handles categorical + non-linear relations)
pk, method = association_matrix(
    df.drop(columns=["Address", "Date"]), interval_cols=num_cols + [TARGET]
)
fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(
    pk,
    cmap="viridis",
    vmin=0,
    vmax=1,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 6},
    ax=ax,
)
ax.set_title(f"{method} association matrix")
plt.show()
display(
    pk[TARGET]
    .drop(TARGET)
    .sort_values(ascending=False)
    .round(3)
    .to_frame(f"{method} with Price")
)

In [ ]:
DF_DY = df.drop(columns=["Address", "Date"])
try:
    from dython.nominal import associations

    assoc = associations(
        DF_DY,
        nominal_columns="auto",
        nom_nom_assoc="theil",
        num_num_assoc="pearson",
        nom_num_assoc="correlation_ratio",
        plot=False,
    )
    a = (
        pd.DataFrame(assoc["corr"])[[TARGET]]
        .drop(TARGET)
        .sort_values(TARGET, ascending=False)
    )
    plt.figure(figsize=(4, 7))
    sns.heatmap(a, annot=True, fmt=".2f", cmap="viridis")
    plt.title(f"Dython associations with {TARGET}")
    plt.show()
except Exception as e:
    print(f"Dython skipped: {type(e).__name__}: {e}")

## 3e) Feature importance

In [ ]:
# Step 3e: four different importance views on the same model matrix
from sklearn.feature_selection import f_regression, mutual_info_regression, RFE
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance

X, y, dropped = make_model_matrix(
    df, TARGET
)  # one-hot for <= 40 levels, median-fill NaN (analysis only)
print("Left out (too many levels):", dropped, "| matrix:", X.shape)
Xs = X.sample(8000, random_state=42)
ys = y.loc[Xs.index]
rf = RandomForestRegressor(200, random_state=42, n_jobs=-1).fit(Xs, ys)
Xt = X.drop(Xs.index)
yt = y.drop(Xs.index)
imp = pd.DataFrame(
    {
        "F_regression": f_regression(Xs, ys)[0],
        "mutual_info": mutual_info_regression(Xs, ys, random_state=42),
        "RF_impurity": rf.feature_importances_,
        "RF_permutation (held-out)": permutation_importance(
            rf, Xt, yt, n_repeats=5, random_state=0, n_jobs=-1
        ).importances_mean,
    },
    index=X.columns,
)
imp["RFE_rank (linear, top 5 = 1)"] = (
    RFE(LinearRegression(), n_features_to_select=5)
    .fit(StandardScaler().fit_transform(Xs), ys)
    .ranking_
)
imp = imp.sort_values("RF_permutation (held-out)", ascending=False)
display(imp.round(3).head(15))
print(f"Random forest R2 on held-out rows: {rf.score(Xt, yt):.3f}")
fig, ax = plt.subplots(1, 3, figsize=(19, 5))
for a, col in zip(ax, ["mutual_info", "RF_impurity", "RF_permutation (held-out)"]):
    imp[col].sort_values().tail(12).plot.barh(ax=a, color="steelblue")
    a.set_title(col)
plt.tight_layout()
plt.show()

## 3f) Other problems

In [ ]:
# Step 3f: the target itself - does the sale Method change what "Price" means?
display(
    df.groupby("Method")[TARGET]
    .agg(["count", "median", "mean"])
    .round(0)
    .assign(
        meaning=lambda t: t.index.map(
            {
                "S": "sold",
                "SP": "sold prior",
                "SA": "sold after auction",
                "PI": "passed in (not sold at auction)",
                "VB": "vendor bid (seller's own bid)",
            }
        )
    )
)

## 4a) Rare combinations

In [ ]:
# Step 4a: rare combinations
combo = df.groupby(["Type", "Regionname", "Method"]).size().sort_values()
print(
    f"{len(combo)} Type x Region x Method combinations exist; {(combo <= 5).sum()} of them have 5 rows or fewer"
)
display(combo.head(8).to_frame("rows"))
rare = pd.Series(
    {
        "1-room house (Type h)": ((df.Rooms == 1) & (df.Type == "h")).sum(),
        "unit with 5+ rooms": ((df.Type == "u") & (df.Rooms >= 5)).sum(),
        "more bathrooms than rooms": (df.Bathroom > df.Rooms).sum(),
        "house in the CBD (Distance < 2 km)": (
            (df.Type == "h") & (df.Distance < 2)
        ).sum(),
        "region outside metro (… Victoria)": df.Regionname.str.contains(
            "Victoria"
        ).sum(),
    }
).to_frame("rows")
display(rare)
try:
    from mlxtend.frequent_patterns import apriori

    basket = pd.get_dummies(df[["Type", "Regionname", "Method"]]).astype(bool)
    freq = apriori(basket, min_support=0.001, use_colnames=True)
    freq["size"] = freq["itemsets"].apply(len)
    display(freq[freq["size"] == 3].sort_values("support").head(8))
except Exception as e:
    print(f"Apriori (mlxtend) skipped: {type(e).__name__}: {e}")

## 4b) Derived features

In [ ]:
# Step 4b: derived features (analysis only - nothing is added to the data yet)
d = df.copy()
d["age_at_sale"] = d["Date"].dt.year - d["YearBuilt"]
d["area_per_room"] = d["BuildingArea"].where(d["BuildingArea"] >= 20) / d["Rooms"]
d["land_per_room"] = d["Landsize"].where(d["Landsize"] > 0) / d["Rooms"]
d["bath_per_room"] = d["Bathroom"] / d["Rooms"]
d["sale_month_index"] = (d["Date"].dt.year - 2016) * 12 + d["Date"].dt.month
new = [
    "age_at_sale",
    "area_per_room",
    "land_per_room",
    "bath_per_room",
    "sale_month_index",
]
display(
    d[new + ["Rooms", "BuildingArea", "YearBuilt"]]
    .corrwith(d[TARGET], method="spearman")
    .sort_values(key=abs, ascending=False)
    .round(3)
    .to_frame("Spearman with Price")
)


## 4c) Trends between support features

In [ ]:
# Step 4c: trends between support features
s3 = df.sample(4000, random_state=3)
fig, axes = plt.subplots(1, 3, figsize=(21, 5.5))
h3 = s3[(s3.Type == "h") & (s3.YearBuilt > 1800)]
sns.scatterplot(data=h3, x="Distance", y="YearBuilt", alpha=0.15, s=8, ax=axes[0])
binned_trend(axes[0], h3["Distance"], h3["YearBuilt"])
axes[0].set_title("Houses: older buildings sit closer to the centre")
sns.scatterplot(
    data=s3[s3.Landsize.between(1, 2000)],
    x="Distance",
    y="Landsize",
    hue="Type",
    alpha=0.3,
    s=8,
    ax=axes[1],
)
axes[1].set_title("Land size grows with distance (Landsize 1-2000 m2)")
sc = axes[2].scatter(
    s3["Longtitude"], s3["Lattitude"], c=np.log10(s3[TARGET]), cmap="viridis", s=6
)
plt.colorbar(sc, ax=axes[2], label="log10(Price)")
axes[2].set_title("Map of sales coloured by price")
plt.tight_layout()
plt.show()
hh = df[(df.Type == "h") & (df.YearBuilt > 1800)]
print(
    "Houses - median YearBuilt by distance band:",
    hh.groupby(pd.cut(hh.Distance, [-1, 3, 6, 10, 15, 50]), observed=True)["YearBuilt"]
    .median()
    .rename(index=str)
    .to_dict(),
)
print(
    "Houses - median price by build decade:",
    hh.groupby((hh.YearBuilt // 10 * 10).clip(1880, 2010))[TARGET]
    .median()
    .round(-4)
    .astype(int)
    .to_dict(),
)

## 4d) Conventional EDA vs tools

In [ ]:
# Step 4d: conventional pandas checks vs. the tools' numbers. Each row is computed two independent ways.
def ydata_says(col, alert):
    hit = yd[(yd.column == col) & (yd.alert == alert)]
    return hit["detail"].iloc[0] if len(hit) else "no alert"


check = pd.DataFrame(
    [
        (
            "BuildingArea missing",
            f"{df.BuildingArea.isna().sum()} ({df.BuildingArea.isna().mean():.1%})",
            ydata_says("BuildingArea", "MISSING"),
        ),
        (
            "Car missing",
            f"{df.Car.isna().sum()} ({df.Car.isna().mean():.1%})",
            ydata_says("Car", "MISSING"),
        ),
        (
            "Landsize zeros",
            f"{(df.Landsize == 0).sum()} ({(df.Landsize == 0).mean():.1%})",
            ydata_says("Landsize", "ZEROS"),
        ),
        (
            "Bathroom zeros",
            f"{(df.Bathroom == 0).sum()} ({(df.Bathroom == 0).mean():.1%})",
            ydata_says("Bathroom", "ZEROS"),
        ),
        (
            "Landsize skew",
            f"{df.Landsize.skew():.1f}",
            ydata_says("Landsize", "SKEWED"),
        ),
        ("Price skew", f"{df.Price.skew():.2f}", ydata_says("Price", "SKEWED")),
        (
            "Rooms ~ Bedroom2",
            f"Pearson {df.Rooms.corr(df.Bedroom2):.2f}",
            ydata_says("Rooms", "HIGH CORRELATION"),
        ),
        (
            "Duplicate rows",
            f"{df.duplicated().sum()}",
            ydata_says("<whole table>", "DUPLICATES"),
        ),
    ],
    columns=["check", "pandas (manual)", "ydata-style rule"],
)
display(check)
# The two calculations must agree on the counts:
assert df.BuildingArea.isna().sum() == miss.loc["BuildingArea", "missing"]
assert (df.Landsize == 0).sum() == column_report(df).loc["Landsize", "zeros"]
print("Manual counts agree with column_report() and the ydata-style rules.")

## 4e) Polynomial trends

In [ ]:
# Step 4e: polynomial trends - how much does a curve add over a straight line?
pairs = [
    ("Distance", None),
    ("YearBuilt", None),
    ("BuildingArea", 0.99),
    ("Landsize", 0.99),
]
fig, axes = plt.subplots(1, 4, figsize=(22, 4.8))
rows = {}
for ax, (x, q) in zip(axes, pairs):
    dd = df[[x, TARGET]].dropna()
    if q:
        dd = dd[(dd[x] > 0) & (dd[x] < dd[x].quantile(q))]
    if x == "YearBuilt":
        dd = dd[dd[x] > 1800]  # leave out the year-1196 typo
    sns.regplot(
        data=dd.sample(min(3000, len(dd)), random_state=1),
        x=x,
        y=TARGET,
        order=3,
        scatter_kws={"alpha": 0.15, "s": 8},
        line_kws={"color": "red"},
        ax=ax,
    )
    rows[x] = poly_r2(dd[x].to_numpy(float), dd[TARGET].to_numpy(float))
    ax.set_title(
        f"{x}: cubic fit" + (f" (0 and top {100 - q * 100:.0f}% removed)" if q else "")
    )
plt.tight_layout()
plt.show()
display(pd.DataFrame(rows).T)

In [ ]:
%pip install "dython<0.7.12" "matplotlib==3.10.0"